# Lab 4 - More classes (F1 teams and drivers)

This notebook presents my solution. The code is in `classes.py` (the `Driver` and `Team` classes) and `main.py` (the `load_teams` function). The data is in `f1_points.csv`. All of these files are in the same folder as this notebook.

## Setup: import the code

In [ ]:
from classes import Driver, Team
from main import load_teams

## Task 1 - the Driver and Team classes

### Driver
A `Driver` has a `name` and `points`. `__repr__` controls what is shown when it is printed.

**Input:** `Driver("Carlos Sainz", 200)`  
**Expected:** `Carlos Sainz (200 pts)`

In [ ]:
sainz = Driver("Carlos Sainz", 200)
print(sainz.name)
print(sainz.points)
print(sainz)

### Team
A `Team` has a `name` and a list of `Driver` objects called `drivers`. The list starts empty and `add_driver` appends to it.

**Input:** a FERRARI team with Carlos Sainz (200) and Charles Leclerc (206)  
**Expected:** 2 drivers, total points 200 + 206 = 406, printed as `FERRARI with drivers Carlos Sainz, Charles Leclerc. Total pts: 406`

In [ ]:
ferrari = Team("FERRARI")
print("drivers before adding:", ferrari.drivers)

leclerc = Driver("Charles Leclerc", 206)
ferrari.add_driver(sainz)
ferrari.add_driver(leclerc)

print("drivers after adding:", ferrari.drivers)
print("total points:", ferrari.get_total_points())
print(ferrari)

### `__lt__` (less than)
One team is "less than" another if it has fewer total points.

**Input:** HAAS with 12 points compared to FERRARI with 406 points  
**Expected:** `haas < ferrari` is `True`, `ferrari < haas` is `False`

In [ ]:
haas = Team("HAAS FERRARI")
haas.add_driver(Driver("Kevin Magnussen", 3))
haas.add_driver(Driver("Nico Hulkenberg", 9))

print(haas.get_total_points(), "<", ferrari.get_total_points())
print("haas < ferrari:", haas < ferrari)
print("ferrari < haas:", ferrari < haas)

## Task 2 - load the drivers and teams from the file

First, what the file looks like (the first 4 lines):

In [ ]:
file = open('f1_points.csv', 'r')
lines = file.readlines()
file.close()

for line in lines[:4]:
    print(line.strip())

How one line is processed: `split(',')` cuts it into 3 parts, and the points are converted from a string to an int.

In [ ]:
line = lines[1].strip()
print("line:", line)

driver_name, team_name, points = line.split(',')
points = int(points)
print("driver_name:", driver_name)
print("team_name:", team_name)
print("points:", points, type(points))

`load_teams` does this for every line. It stores the teams in a dictionary (key = team name, value = `Team` object). Before adding a driver it checks `if team_name not in teams`, so only **one** `Team` object is created for each team even though each team is in the file more than once.

**Input:** `f1_points.csv` (22 drivers)  
**Expected:** 10 teams, each with 2 or more drivers

In [ ]:
teams = load_teams('f1_points.csv')

print("type of collection:", type(teams))
print("number of teams:", len(teams))
print()
for team_name in teams:
    print(team_name, "->", len(teams[team_name].drivers), "drivers")

Check one team against the file by hand: FERRARI has Carlos Sainz (200) and Charles Leclerc (206).

**Expected:** total 406

In [ ]:
print(teams['FERRARI'])
print(teams['FERRARI'].drivers)

Check that no driver was lost or counted twice.

**Expected:** 22 drivers in total (the file has 22 data lines)

In [ ]:
driver_count = 0
for team in teams.values():
    driver_count += len(team.drivers)

print("drivers in file:", len(lines) - 1)
print("drivers in teams:", driver_count)

## Task 3 - sort the teams by points

`sorted` compares the `Team` objects using `__lt__`, so the team with the fewest points comes first.

**Expected:** HAAS FERRARI (12) first, RED BULL RACING HONDA RBPT (860) last

In [ ]:
sorted_teams = sorted(teams.values())

for team in sorted_teams:
    print(team)

Most points first, using `reverse=True`:

In [ ]:
position = 1
for team in sorted(teams.values(), reverse=True):
    print(position, team.name, team.get_total_points())
    position += 1

## Test cases

### Normal case: a small list of teams I can check by hand

**Input:** teams with 30, 10 and 20 points  
**Expected order:** B (10), C (20), A (30)

In [ ]:
team_a = Team("A")
team_a.add_driver(Driver("a1", 10))
team_a.add_driver(Driver("a2", 20))

team_b = Team("B")
team_b.add_driver(Driver("b1", 4))
team_b.add_driver(Driver("b2", 6))

team_c = Team("C")
team_c.add_driver(Driver("c1", 20))
team_c.add_driver(Driver("c2", 0))

for team in sorted([team_a, team_b, team_c]):
    print(team)

### Special case: a team with no drivers

**Input:** `Team("NEW TEAM")` with nothing added  
**Expected:** total points 0, and it sorts before every team that has points

In [ ]:
empty_team = Team("NEW TEAM")
print(empty_team)
print("total:", empty_team.get_total_points())
print("empty_team < team_b:", empty_team < team_b)

### Special case: a driver with 0 points

**Input:** Nyck De Vries scored 0 points in the file  
**Expected:** he is still added to his team, and the team total does not change because of him (6 + 2 + 0 + 17 = 25)

In [ ]:
alphatauri = teams['ALPHATAURI HONDA RBPT']
print(alphatauri.drivers)
print("total:", alphatauri.get_total_points())

### Special case: a team with more than 2 drivers

**Input:** ALPHATAURI HONDA RBPT appears 4 times in the file  
**Expected:** one `Team` object with 4 drivers

In [ ]:
print("number of drivers:", len(alphatauri.drivers))

### Special case: two teams with the same points

**Input:** two teams with 10 points each  
**Expected:** neither one is less than the other, so both comparisons are `False`

In [ ]:
tie_1 = Team("TIE 1")
tie_1.add_driver(Driver("t1", 10))
tie_2 = Team("TIE 2")
tie_2.add_driver(Driver("t2", 10))

print("tie_1 < tie_2:", tie_1 < tie_2)
print("tie_2 < tie_1:", tie_2 < tie_1)
print(sorted([tie_1, tie_2]))

### Error case: adding something that is not a Driver

**Input:** `add_driver("Max Verstappen")` (a string, not a `Driver` object)  
**Expected:** `add_driver` raises a `TypeError`, and nothing is added to the team

In [ ]:
test_team = Team("TEST")
try:
    test_team.add_driver("Max Verstappen")
except TypeError as error:
    print("TypeError:", error)

print("drivers:", test_team.drivers)

### Error case: bad lines in the file

**Input:** `test_bad_data.csv`, which has a line where the points are `abc`, a blank line, and a line with only 2 columns  
**Expected:** the bad lines are skipped with a message, the good lines are still loaded (TEAM A = 10, TEAM B = 5)

In [ ]:
file = open('test_bad_data.csv', 'r')
print(file.read())
file.close()

In [ ]:
test_teams = load_teams('test_bad_data.csv')
print()
for team in sorted(test_teams.values()):
    print(team)

### Error case: the file does not exist

**Input:** `load_teams('no_such_file.csv')`  
**Expected:** Python raises a `FileNotFoundError`, which can be caught with try/except

In [ ]:
try:
    load_teams('no_such_file.csv')
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

## Summary

- **Task 1:** `Driver` and `Team` follow the UML. `Team` keeps a list of `Driver` objects, and `__repr__` and `__lt__` work as shown above.
- **Task 2:** `load_teams` reads `f1_points.csv` and builds 10 `Team` objects (one per team) holding 22 `Driver` objects in total.
- **Task 3:** `sorted` puts the teams in order of total points using `__lt__`.